In [ ]:
# DIAGNOSTIC CELL: Run this to check your environment
import sys
print(f"Python Executable: {sys.executable}")
try:
    import numpy as np
    print(f"Numpy version: {np.__version__}")
    print("Numpy is working correctly!")
except ImportError:
    print("Numpy not found. Attempting to install in the current kernel...")
    %pip install numpy matplotlib seaborn pandas openpyxl
    print("Installation complete. PLEASE RESTART THE KERNEL (Kernel -> Restart) then run the cells again.")

# Dynamic Programming and Model-Free Learning: The Race Car Problem

This notebook demonstrates the full spectrum of Reinforcement Learning algorithms applied to the **Race Car MDP**:
1. **Planning (Dynamic Programming)**: Policy Evaluation, Policy Iteration, Value Iteration.
2. **Learning (Model-Free)**: Monte Carlo, TD(0), and Q-Learning.

## 1. Problem Definition

### States ($S$)
- **Cool (C)**: Engine is running at optimal temperature.
- **Warm (W)**: Engine is warm.
- **Overheated (OH)**: Engine has failed (Terminal state).

### Actions ($A$)
- **Slow (S)**: Safe speed. Lower reward, low risk.
- **Fast (F)**: High speed. Higher reward, high risk.

### Complete Model Dynamics $p(s', r | s, a)$

| Current State ($s$) | Action ($a$) | Next State ($s'$) | Reward ($r$) | Probability ($p$) |
|:---|:---|:---|:---|:---|
| Cool | Slow | Cool | +1.0 | 0.5 |
| Cool | Slow | Warm | +1.0 | 0.5 |
| Cool | Fast | Cool | +2.0 | 0.5 |
| Cool | Fast | Warm | +2.0 | 0.5 |
| Warm | Slow | Cool | +1.0 | 0.5 |
| Warm | Slow | Warm | +1.0 | 0.5 |
| Warm | Fast | Overheated | -10.0 | 1.0 |
| Overheated | *Any* | Overheated | 0.0 | 1.0 |

### Parameters
- Discount Factor $\gamma = 0.9$

## The `P` Data Structure (Transition Model)

The entire MDP dynamics are encoded in a nested dictionary `P` that follows the **OpenAI Gym convention**:

```
P[state][action] = [(probability, next_state, reward, is_terminal), ...]
```

Each entry is a **list of tuples**, where each tuple represents one possible outcome of taking that action in that state. The list contains multiple tuples when the transition is **stochastic** (e.g., going Fast from Cool can land you in Cool or Warm with equal probability).

### Tuple Fields

| Field | Type | Meaning |
|:------|:-----|:--------|
| `probability` | float | $p(s', r \| s, a)$ — chance this outcome occurs. All probabilities for a given (s, a) sum to 1.0 |
| `next_state` | int | $s'$ — the state the agent transitions to |
| `reward` | float | $r$ — immediate reward received on this transition |
| `is_terminal` | bool | Whether $s'$ is a terminal/absorbing state (episode ends) |

### Concrete Example from This MDP

```python
P[COOL][FAST] = [(0.5, COOL, 2.0, False),   # 50% chance: stay Cool, earn +2, continue
                 (0.5, WARM, 2.0, False)]    # 50% chance: become Warm, earn +2, continue

P[WARM][FAST] = [(1.0, OVERHEATED, -10.0, True)]  # 100% chance: overheat, lose 10, episode ends
```

### Why This Structure?

This format directly supports the Bellman equation computation. For any value update, we iterate over all possible outcomes:

$$V(s) = \sum_a \pi(a|s) \sum_{s', r} p(s', r | s, a) \left[ r + \gamma V(s') \right]$$

Each tuple `(p, ns, r, done)` in `P[s][a]` gives us one term in the inner summation $\sum_{s',r}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass

# Constants
COOL, WARM, OVERHEATED = 0, 1, 2
STATE_NAMES = {COOL: "Cool", WARM: "Warm", OVERHEATED: "Overheated"}
SLOW, FAST = 0, 1
ACTION_NAMES = {SLOW: "Slow", FAST: "Fast"}

@dataclass
class RaceCarMDP:
    num_states: int = 3
    num_actions: int = 2
    gamma: float = 0.9
    def __post_init__(self):
        self.P = {
            COOL: { SLOW: [(0.5, COOL, 1.0, False), (0.5, WARM, 1.0, False)],
                    FAST: [(0.5, COOL, 2.0, False), (0.5, WARM, 2.0, False)] },
            WARM: { SLOW: [(0.5, COOL, 1.0, False), (0.5, WARM, 1.0, False)],
                    FAST: [(1.0, OVERHEATED, -10.0, True)] },
            OVERHEATED: { SLOW: [(1.0, OVERHEATED, 0.0, True)],
                          FAST: [(1.0, OVERHEATED, 0.0, True)] }
        }

mdp = RaceCarMDP()

## 2. Policy Evaluation (Prediction)

**Goal:** Given a fixed policy $\pi$, compute how good each state is — i.e., find $V_\pi(s)$.

### Algorithm Flow

```
┌─────────────────────────────────────────────────────────┐
│  POLICY EVALUATION                                      │
│                                                         │
│  Input: policy π, MDP (P, γ)                           │
│  Output: V_π(s) for all states                         │
│                                                         │
│  V(s) ← 0 for all s                                   │
│                                                         │
│  ┌───── REPEAT (sweep) ─────────────────────────────┐  │
│  │                                                   │  │
│  │  For EACH state s:                               │  │
│  │    v_new ← Σ_a π(a|s) · Σ_{s',r} p(s',r|s,a)  │  │
│  │                         · [r + γ·V(s')]          │  │
│  │    V(s) ← v_new                                  │  │
│  │                                                   │  │
│  │  Check: max change (delta) < θ?                  │  │
│  │    YES → STOP (converged)                        │  │
│  │    NO  → another sweep                           │  │
│  └───────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────┘
```

### What Happens in ONE Sweep (for state Cool, random policy)

For state Cool with equiprobable policy $\pi(\text{Slow}|\text{Cool}) = 0.5$, $\pi(\text{Fast}|\text{Cool}) = 0.5$:

$$V(\text{Cool}) = \underbrace{0.5}_{\pi(S|C)} \cdot \left[ \underbrace{0.5 \cdot (1 + 0.9 \cdot V(C))}_{P[C][S][0]} + \underbrace{0.5 \cdot (1 + 0.9 \cdot V(W))}_{P[C][S][1]} \right] + \underbrace{0.5}_{\pi(F|C)} \cdot \left[ \underbrace{0.5 \cdot (2 + 0.9 \cdot V(C))}_{P[C][F][0]} + \underbrace{0.5 \cdot (2 + 0.9 \cdot V(W))}_{P[C][F][1]} \right]$$

Initially with V = [0, 0, 0]:
$$V(\text{Cool}) = 0.5 \cdot [0.5 \cdot 1 + 0.5 \cdot 1] + 0.5 \cdot [0.5 \cdot 2 + 0.5 \cdot 2] = 0.5 \cdot 1 + 0.5 \cdot 2 = 1.5$$

Each sweep refines V using the previous sweep's estimates until convergence.

### Initial Random Policy (Equiprobable)

| State | $\pi$(Slow) | $\pi$(Fast) |
| :--- | :---: | :---: |
| **Cool** | 0.5 | 0.5 |
| **Warm** | 0.5 | 0.5 |
| **Overheated** | 0.5 | 0.5 |

In [ ]:
def policy_evaluation(mdp, policy, theta=1e-8):
    V = np.zeros(mdp.num_states)
    while True:
        delta = 0
        for s in range(mdp.num_states):
            v_new = 0
            for a, prob_a in enumerate(policy[s]):
                for prob, ns, r, done in mdp.P[s][a]:
                    v_new += prob_a * prob * (r + mdp.gamma * V[ns] * (not done))
            delta = max(delta, np.abs(v_new - V[s]))
            V[s] = v_new
        if delta < theta: break
    return V

random_policy = np.ones((mdp.num_states, mdp.num_actions)) / mdp.num_actions
V_random = policy_evaluation(mdp, random_policy)
print("Value Function for Random Policy:")
for s in range(mdp.num_states): print(f"{STATE_NAMES[s]}: {V_random[s]:.4f}")

In [ ]:
# DETAILED EXECUTION TRACE: Policy Evaluation
# Watch the first 3 sweeps to see how V converges under the random policy

def policy_evaluation_traced(mdp, policy, max_sweeps=3):
    """Policy Evaluation with step-by-step trace of first few sweeps."""
    V = np.zeros(mdp.num_states)
    print("=" * 70)
    print("POLICY EVALUATION — DETAILED TRACE")
    print("=" * 70)
    print(f"\nInitial V = [Cool=0, Warm=0, OH=0]")
    print(f"Policy: equiprobable (0.5 Slow, 0.5 Fast for each state)")
    print(f"γ = {mdp.gamma}\n")
    
    for sweep in range(max_sweeps):
        print(f"{'─' * 70}")
        print(f"SWEEP {sweep + 1}")
        print(f"{'─' * 70}")
        delta = 0
        for s in range(mdp.num_states):
            v_old = V[s]
            v_new = 0
            contributions = []
            for a, prob_a in enumerate(policy[s]):
                for prob, ns, r, done in mdp.P[s][a]:
                    contrib = prob_a * prob * (r + mdp.gamma * V[ns] * (not done))
                    v_new += contrib
                    contributions.append(
                        f"    π({ACTION_NAMES[a]}|{STATE_NAMES[s]})={prob_a:.1f} × "
                        f"P(→{STATE_NAMES[ns]})={prob:.1f} × "
                        f"(r={r:+.1f} + γ·V({STATE_NAMES[ns]})={mdp.gamma:.1f}×{V[ns]:.4f}) "
                        f"= {contrib:.4f}"
                    )
            delta = max(delta, abs(v_new - v_old))
            V[s] = v_new
            print(f"\n  State: {STATE_NAMES[s]} (V_old = {v_old:.4f})")
            for c in contributions:
                print(c)
            print(f"  → V({STATE_NAMES[s]}) = {v_new:.4f}  (Δ = {abs(v_new - v_old):.4f})")
        
        print(f"\n  Sweep {sweep + 1} complete: V = [Cool={V[COOL]:.4f}, Warm={V[WARM]:.4f}, OH={V[OVERHEATED]:.4f}]")
        print(f"  Max delta this sweep = {delta:.6f}")
    
    print(f"\n{'═' * 70}")
    print(f"After full convergence ({V_random[COOL]:.4f} takes many more sweeps):")
    print(f"  V(Cool) = {V_random[COOL]:.4f}, V(Warm) = {V_random[WARM]:.4f}")
    print(f"{'═' * 70}")

policy_evaluation_traced(mdp, random_policy, max_sweeps=3)

## 3. Policy Iteration (Planning)

**Goal:** Find the optimal policy $\pi^*$ by alternating between evaluating the current policy and improving it.

### Algorithm Flow

```
┌──────────────────────────────────────────────────────────────────────┐
│  POLICY ITERATION                                                    │
│                                                                      │
│  Initialize: π(s) = Slow for all s (arbitrary deterministic policy) │
│                                                                      │
│  ┌───── REPEAT ──────────────────────────────────────────────────┐  │
│  │                                                                │  │
│  │  ┌─ STEP 1: POLICY EVALUATION ──────────────────────────────┐ │  │
│  │  │  Run full iterative policy evaluation to get V_π(s)      │ │  │
│  │  │  (This is the INNER loop — many sweeps until convergence)│ │  │
│  │  └──────────────────────────────────────────────────────────┘ │  │
│  │                          ↓                                     │  │
│  │  ┌─ STEP 2: POLICY IMPROVEMENT ─────────────────────────────┐ │  │
│  │  │  For each state s:                                        │ │  │
│  │  │    Compute Q(s,a) for ALL actions a                       │ │  │
│  │  │    π(s) ← argmax_a Q(s,a)   [pick the BEST action]       │ │  │
│  │  └──────────────────────────────────────────────────────────┘ │  │
│  │                          ↓                                     │  │
│  │  Policy changed?                                               │  │
│  │    YES → go back to Step 1                                    │  │
│  │    NO  → STOP (π is optimal!)                                 │  │
│  └────────────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────────────┘
```

### Key Insight: Why Does This Work?

The improvement step uses **action-value function** $Q_\pi(s, a)$:

$$Q_\pi(s, a) = \sum_{s', r} p(s', r | s, a) \left[ r + \gamma V_\pi(s') \right]$$

This tells us: "If I take action $a$ in state $s$ and then follow policy $\pi$ forever after, what's my expected return?"

By picking $\text{argmax}_a Q_\pi(s, a)$, we guarantee the new policy is **at least as good** as the old one (Policy Improvement Theorem).

$$\pi'(s) = \text{argmax}_a Q_\pi(s, a)$$

In [ ]:
def policy_iteration(mdp):
    policy = np.zeros((mdp.num_states, mdp.num_actions)); policy[:, SLOW] = 1.0
    V = np.zeros(mdp.num_states)
    p_hist, v_hist = [policy.copy()], [V.copy()]
    
    while True:
        V = policy_evaluation(mdp, policy)
        policy_stable = True
        new_p = policy.copy()
        for s in range(mdp.num_states):
            old_a = np.argmax(policy[s])
            q = [sum(p*(r + mdp.gamma*V[ns]*(not d)) for p, ns, r, d in mdp.P[s][a]) for a in range(mdp.num_actions)]
            best_a = np.argmax(q)
            if old_a != best_a:
                policy_stable = False
                new_p[s] = np.eye(mdp.num_actions)[best_a]
        v_hist.append(V.copy())
        if not policy_stable: policy = new_p.copy(); p_hist.append(policy.copy())
        else: break
    return policy, V, p_hist, v_hist

opt_p, V_opt, pi_h, v_h = policy_iteration(mdp)
print("Policy Iteration History:")
for i in range(len(pi_h)):
    print(f"\n--- Step {i} ---")
    data = [[STATE_NAMES[s], pi_h[i][s][SLOW], pi_h[i][s][FAST], v_h[i][s]] for s in range(mdp.num_states)]
    print(pd.DataFrame(data, columns=['State', 'pi(S)', 'pi(F)', 'V(s)']).to_string(index=False))
    print("(Initialization)" if i == 0 else "(Improved)")

print("\n--- FINAL CONCLUSION: OPTIMAL STRATEGY ---")
final_data = [[STATE_NAMES[s], ACTION_NAMES[np.argmax(opt_p[s])] if s != OVERHEATED else "Terminal"] for s in range(mdp.num_states)]
print(pd.DataFrame(final_data, columns=['State', 'Action']).to_string(index=False))

In [ ]:
# DETAILED EXECUTION TRACE: Policy Iteration
# Watch how the algorithm flips the policy from "always Slow" to "Fast when Cool"

def policy_iteration_traced(mdp):
    """Policy Iteration with step-by-step trace showing evaluation + improvement."""
    policy = np.zeros((mdp.num_states, mdp.num_actions))
    policy[:, SLOW] = 1.0  # Start: always go Slow
    
    print("=" * 70)
    print("POLICY ITERATION — DETAILED TRACE")
    print("=" * 70)
    print(f"\nInitial policy: π(Cool)=Slow, π(Warm)=Slow, π(OH)=Slow")
    print(f"γ = {mdp.gamma}\n")
    
    iteration = 0
    while True:
        iteration += 1
        print(f"{'═' * 70}")
        print(f"ITERATION {iteration}")
        print(f"{'═' * 70}")
        
        # Step 1: Policy Evaluation
        V = policy_evaluation(mdp, policy)
        print(f"\n  STEP 1 — POLICY EVALUATION (converged):")
        print(f"  Current policy: Cool→{ACTION_NAMES[np.argmax(policy[COOL])]}, "
              f"Warm→{ACTION_NAMES[np.argmax(policy[WARM])]}")
        print(f"  V_π = [Cool={V[COOL]:.4f}, Warm={V[WARM]:.4f}, OH={V[OVERHEATED]:.4f}]")
        
        # Step 2: Policy Improvement
        print(f"\n  STEP 2 — POLICY IMPROVEMENT:")
        policy_stable = True
        new_policy = policy.copy()
        
        for s in range(mdp.num_states):
            old_a = np.argmax(policy[s])
            q_values = []
            for a in range(mdp.num_actions):
                q = sum(p * (r + mdp.gamma * V[ns] * (not d)) 
                        for p, ns, r, d in mdp.P[s][a])
                q_values.append(q)
                print(f"    Q({STATE_NAMES[s]}, {ACTION_NAMES[a]}) = ", end="")
                terms = []
                for p, ns, r, d in mdp.P[s][a]:
                    terms.append(f"{p}×({r} + {mdp.gamma}×{V[ns]:.4f}×{int(not d)})")
                print(" + ".join(terms) + f" = {q:.4f}")
            
            best_a = np.argmax(q_values)
            if old_a != best_a:
                policy_stable = False
                new_policy[s] = np.eye(mdp.num_actions)[best_a]
                print(f"    → π({STATE_NAMES[s]}): {ACTION_NAMES[old_a]} → {ACTION_NAMES[best_a]} ✗ CHANGED")
            else:
                print(f"    → π({STATE_NAMES[s]}): {ACTION_NAMES[old_a]} (unchanged) ✓")
        
        policy = new_policy.copy()
        
        if policy_stable:
            print(f"\n  Policy STABLE — no changes made. Algorithm CONVERGED!")
            break
        else:
            print(f"\n  Policy changed — going back to evaluation...")
    
    print(f"\n{'═' * 70}")
    print(f"OPTIMAL POLICY FOUND:")
    print(f"  Cool → {ACTION_NAMES[np.argmax(policy[COOL])]}")
    print(f"  Warm → {ACTION_NAMES[np.argmax(policy[WARM])]}")
    print(f"{'═' * 70}")

policy_iteration_traced(mdp)

## 4. Value Iteration (Shortcut Planning)

Unlike Policy Iteration (which alternates between full policy evaluation and improvement), Value Iteration combines both into a single update by using `max` directly in the Bellman equation:

$$V(s) \leftarrow \max_a \sum_{s', r} p(s', r | s, a) \left[ r + \gamma V(s') \right]$$

### Anatomy of the Nested Loops

The code below has **4 levels of nesting**. Here's what each level does:

```
while True:                          ← LEVEL 1: Sweep loop (repeat until convergence)
    for s in range(num_states):      ← LEVEL 2: Visit every state in one sweep
        q = [                        ← LEVEL 3: Compute Q(s,a) for EACH action
            sum(                      ← LEVEL 4: Sum over all stochastic outcomes
                p * (r + γ*V[ns]*(not done))
                for p, ns, r, done in P[s][a]
            )
            for a in range(num_actions)
        ]
        V[s] = max(q)                ← Take the best action's value (the "max" in Bellman optimality)
```

### Level-by-Level Breakdown

| Level | Loop | What it computes | Maps to math |
|:------|:-----|:-----------------|:-------------|
| 1 | `while True` | Repeats full sweeps until $\max_s \|V_{new}(s) - V_{old}(s)\| < \theta$ | Iterative convergence |
| 2 | `for s in range(num_states)` | Updates V for each state in one sweep | The $\forall s$ in the update rule |
| 3 | `for a in range(num_actions)` | Builds a list of Q-values, one per action | The $\max_a$ operator needs all Q(s,a) values |
| 4 | `for p, ns, r, done in P[s][a]` | Sums over all possible transitions for one (s, a) pair | $\sum_{s',r} p(s',r\|s,a)[r + \gamma V(s')]$ |

### The `(not done)` Trick

When `done=True`, the next state is terminal — its future value is 0 by definition. Multiplying by `(not done)` (which is `0` for terminal states) implements this without needing a special case:

$$r + \gamma \cdot V(s') \cdot \mathbb{1}[s' \text{ is not terminal}]$$

### Convergence

Each sweep brings V closer to $V^*$. The algorithm stops when the largest change across all states (`delta`) falls below a threshold (`1e-8`), meaning V has effectively converged to the optimal value function.

In [ ]:
def value_iteration(mdp):
    V = np.zeros(mdp.num_states)
    v_hist = [V.copy()]
    while True:
        delta = 0
        new_V = V.copy()
        for s in range(mdp.num_states):
            q = [sum(p*(r + mdp.gamma*V[ns]*(not d)) for p, ns, r, d in mdp.P[s][a]) for a in range(mdp.num_actions)]
            new_V[s] = max(q)
            delta = max(delta, abs(new_V[s] - V[s]))
        V = new_V; v_hist.append(V.copy())
        if delta < 1e-8: break
    return V, v_hist

V_vi, vi_h = value_iteration(mdp)
print("Value Iteration History (Select Sweeps):")
for i in [0, 1, 2, 5, 10, len(vi_h)-1]:
    if i >= len(vi_h): continue
    print(f"Sweep {i}: V(Cool)={vi_h[i][COOL]:.2f}, V(Warm)={vi_h[i][WARM]:.2f}")

In [ ]:
# DETAILED EXECUTION TRACE: Value Iteration
# Watch how V converges to optimal values in each sweep

def value_iteration_traced(mdp, max_sweeps=4):
    """Value Iteration with step-by-step trace showing Q-value computation per state."""
    V = np.zeros(mdp.num_states)
    
    print("=" * 70)
    print("VALUE ITERATION — DETAILED TRACE")
    print("=" * 70)
    print(f"\nInitial V = [Cool=0, Warm=0, OH=0]")
    print(f"γ = {mdp.gamma}")
    print(f"\nKey difference from Policy Iteration:")
    print(f"  - No separate policy evaluation (no inner convergence loop)")
    print(f"  - Each sweep directly takes max over actions (Bellman optimality)\n")
    
    for sweep in range(1, max_sweeps + 1):
        print(f"{'═' * 70}")
        print(f"SWEEP {sweep}")
        print(f"{'═' * 70}")
        delta = 0
        new_V = V.copy()
        
        for s in range(mdp.num_states):
            q_values = []
            print(f"\n  State: {STATE_NAMES[s]} (V_old = {V[s]:.4f})")
            for a in range(mdp.num_actions):
                q = sum(p * (r + mdp.gamma * V[ns] * (not d)) 
                        for p, ns, r, d in mdp.P[s][a])
                q_values.append(q)
                terms = []
                for p, ns, r, d in mdp.P[s][a]:
                    terms.append(f"{p}×({r:+.1f} + {mdp.gamma}×{V[ns]:.4f}×{int(not d)})")
                print(f"    Q(·, {ACTION_NAMES[a]}) = {' + '.join(terms)} = {q:.4f}")
            
            new_V[s] = max(q_values)
            best_a = np.argmax(q_values)
            delta = max(delta, abs(new_V[s] - V[s]))
            print(f"    → V({STATE_NAMES[s]}) = max({q_values[0]:.4f}, {q_values[1]:.4f}) "
                  f"= {new_V[s]:.4f} [best: {ACTION_NAMES[best_a]}]")
        
        V = new_V
        print(f"\n  Sweep {sweep} result: V = [Cool={V[COOL]:.4f}, Warm={V[WARM]:.4f}, OH={V[OVERHEATED]:.4f}]")
        print(f"  Max delta = {delta:.6f}")
    
    print(f"\n{'═' * 70}")
    print(f"VALUE ITERATION vs POLICY ITERATION COMPARISON:")
    print(f"  Value Iteration converged V:  Cool={V_vi[COOL]:.4f}, Warm={V_vi[WARM]:.4f}")
    print(f"  Policy Iteration converged V: Cool={V_opt[COOL]:.4f}, Warm={V_opt[WARM]:.4f}")
    print(f"  (They converge to the SAME optimal values!)")
    print(f"{'═' * 70}")

value_iteration_traced(mdp, max_sweeps=3)

## 5. Asynchronous Dynamic Programming

**Goal:** Same as synchronous DP (find $V^*$ or $\pi^*$), but **without sweeping all states in a fixed order**. Instead, update states in any order — prioritizing states that matter most.

### Why Asynchronous?

In synchronous DP (Sections 2-4), every sweep visits **all** states exactly once. This is wasteful:
- Some states are already near-converged (e.g., Overheated is always 0)
- Some states change rapidly and need more attention
- In large MDPs (millions of states), a full sweep is expensive

Asynchronous DP removes the requirement of complete sweeps. Any state can be updated at any time, in any order, as many times as needed.

### Variants of Asynchronous DP

```
┌──────────────────────────────────────────────────────────────────────┐
│  ASYNCHRONOUS DP VARIANTS                                            │
│                                                                      │
│  1. IN-PLACE DP                                                      │
│     Update V(s) immediately (don't keep old V separate)              │
│     Subsequent states in the same "sweep" see the new value          │
│     → Faster propagation of information                              │
│                                                                      │
│  2. PRIORITIZED SWEEPING                                             │
│     Maintain a priority queue of states                              │
│     Priority = magnitude of Bellman error |V_new(s) - V_old(s)|     │
│     Always update the state with the LARGEST error first             │
│     → Focus computation where it helps most                          │
│                                                                      │
│  3. REAL-TIME DP                                                     │
│     Only update states that the agent ACTUALLY visits                │
│     States the agent never reaches are never computed                │
│     → Natural for online/embedded systems                            │
└──────────────────────────────────────────────────────────────────────┘
```

### Key Difference: Synchronous vs Asynchronous

```
SYNCHRONOUS (Value Iteration):           ASYNCHRONOUS (Prioritized Sweeping):
─────────────────────────────           ───────────────────────────────────

Sweep 1: Update Cool                    Step 1: Update Warm (biggest error)
          Update Warm                    Step 2: Update Cool (affected by Warm)
          Update Overheated             Step 3: Update Cool again (still changing)
                                         Step 4: Update Warm (check if stable)
Sweep 2: Update Cool                    Step 5: DONE (both converged!)
          Update Warm                    
          Update Overheated             Skipped Overheated entirely!
                                         (its Bellman error was always 0)
Sweep 3: Update Cool
          ...
```

### Convergence Guarantee

Asynchronous DP still converges to $V^*$ as long as:
1. Every state continues to be updated (no state is permanently ignored)
2. The updates use the Bellman equation correctly

The ORDER and FREQUENCY of updates don't matter for correctness — only efficiency.

### Implementations Below

We implement three variants:
1. **In-Place Value Iteration** — uses updated values immediately within the same pass
2. **Prioritized Sweeping** — uses a priority queue to focus on high-error states
3. **Real-Time DP** — updates only states visited during simulated trajectories

In [ ]:
import heapq

# ═══════════════════════════════════════════════════════════════════════
# VARIANT 1: IN-PLACE VALUE ITERATION
# ═══════════════════════════════════════════════════════════════════════

def inplace_value_iteration(mdp, theta=1e-8):
    """In-place: updates V(s) immediately, so later states in the same
    pass already see the new value. No separate old_V / new_V arrays."""
    V = np.zeros(mdp.num_states)
    sweeps = 0
    while True:
        delta = 0
        sweeps += 1
        for s in range(mdp.num_states):
            v_old = V[s]
            V[s] = max(
                sum(p * (r + mdp.gamma * V[ns] * (not d))
                    for p, ns, r, d in mdp.P[s][a])
                for a in range(mdp.num_actions)
            )
            delta = max(delta, abs(V[s] - v_old))
        if delta < theta:
            break
    return V, sweeps

# ═══════════════════════════════════════════════════════════════════════
# VARIANT 2: PRIORITIZED SWEEPING
# ═══════════════════════════════════════════════════════════════════════

def prioritized_sweeping(mdp, theta=1e-8):
    """Priority queue based: always update the state with the largest
    Bellman error. After updating s, check predecessors for new errors."""
    V = np.zeros(mdp.num_states)
    
    # Build predecessor map: which states can transition INTO state s?
    predecessors = {s: set() for s in range(mdp.num_states)}
    for s in range(mdp.num_states):
        for a in range(mdp.num_actions):
            for p, ns, r, d in mdp.P[s][a]:
                if p > 0:
                    predecessors[ns].add(s)
    
    # Initialize priority queue with Bellman errors for all states
    # Using negative priority because heapq is a min-heap
    pq = []
    for s in range(mdp.num_states):
        bellman_val = max(
            sum(p * (r + mdp.gamma * V[ns] * (not d))
                for p, ns, r, d in mdp.P[s][a])
            for a in range(mdp.num_actions)
        )
        error = abs(bellman_val - V[s])
        if error > theta:
            heapq.heappush(pq, (-error, s))
    
    updates = 0
    update_log = []
    
    while pq:
        neg_priority, s = heapq.heappop(pq)
        
        # Update V(s)
        v_old = V[s]
        V[s] = max(
            sum(p * (r + mdp.gamma * V[ns] * (not d))
                for p, ns, r, d in mdp.P[s][a])
            for a in range(mdp.num_actions)
        )
        updates += 1
        update_log.append((s, v_old, V[s], -neg_priority))
        
        # Check predecessors — do they now have a significant error?
        for pred in predecessors[s]:
            bellman_val = max(
                sum(p * (r + mdp.gamma * V[ns] * (not d))
                    for p, ns, r, d in mdp.P[pred][a])
                for a in range(mdp.num_actions)
            )
            error = abs(bellman_val - V[pred])
            if error > theta:
                heapq.heappush(pq, (-error, pred))
    
    return V, updates, update_log

# ═══════════════════════════════════════════════════════════════════════
# VARIANT 3: REAL-TIME DP
# ═══════════════════════════════════════════════════════════════════════

def realtime_dp(mdp, num_episodes=50, theta=1e-8):
    """Real-Time DP: only update states the agent actually visits
    while following a greedy policy w.r.t. current V."""
    np.random.seed(123)
    V = np.zeros(mdp.num_states)
    total_updates = 0
    states_updated = set()
    
    for ep in range(num_episodes):
        s = COOL
        while True:
            # Update V(s) for current state (the key Async DP step)
            V[s] = max(
                sum(p * (r + mdp.gamma * V[ns] * (not d))
                    for p, ns, r, d in mdp.P[s][a])
                for a in range(mdp.num_actions)
            )
            total_updates += 1
            states_updated.add(s)
            
            # Act greedily w.r.t. current V
            q_vals = [sum(p * (r + mdp.gamma * V[ns] * (not d))
                         for p, ns, r, d in mdp.P[s][a])
                      for a in range(mdp.num_actions)]
            best_a = np.argmax(q_vals)
            
            # Simulate transition
            ns, r, done = step(mdp, s, best_a)
            if done:
                break
            s = ns
    
    return V, total_updates, states_updated

# ═══════════════════════════════════════════════════════════════════════
# RUN ALL THREE AND COMPARE
# ═══════════════════════════════════════════════════════════════════════

print("=" * 70)
print("ASYNCHRONOUS DP — ALL THREE VARIANTS")
print("=" * 70)

# 1. In-Place
V_ip, sweeps_ip = inplace_value_iteration(mdp)
print(f"\n{'─' * 70}")
print(f"VARIANT 1: IN-PLACE VALUE ITERATION")
print(f"{'─' * 70}")
print(f"  Converged in {sweeps_ip} sweeps")
print(f"  V* = [Cool={V_ip[COOL]:.4f}, Warm={V_ip[WARM]:.4f}, OH={V_ip[OVERHEATED]:.4f}]")
print(f"  (Synchronous Value Iteration took {len(vi_h)-1} sweeps)")
print(f"  Speedup: In-place often converges faster because updated values")
print(f"  propagate within the same sweep (Gauss-Seidel style)")

# 2. Prioritized Sweeping
V_ps, updates_ps, log_ps = prioritized_sweeping(mdp)
print(f"\n{'─' * 70}")
print(f"VARIANT 2: PRIORITIZED SWEEPING")
print(f"{'─' * 70}")
print(f"  Converged in {updates_ps} individual state updates")
print(f"  V* = [Cool={V_ps[COOL]:.4f}, Warm={V_ps[WARM]:.4f}, OH={V_ps[OVERHEATED]:.4f}]")
print(f"\n  Update sequence (state, V_old → V_new, priority):")
for s, v_old, v_new, priority in log_ps[:10]:
    print(f"    {STATE_NAMES[s]:>10}: {v_old:.4f} → {v_new:.4f}  (priority={priority:.4f})")
if len(log_ps) > 10:
    print(f"    ... ({len(log_ps) - 10} more updates)")
print(f"\n  Key insight: Overheated was {'NEVER' if all(s != OVERHEATED for s, _, _, _ in log_ps) else 'rarely'} "
      f"updated (Bellman error is always 0)")

# 3. Real-Time DP
V_rt, updates_rt, visited_rt = realtime_dp(mdp, num_episodes=50)
print(f"\n{'─' * 70}")
print(f"VARIANT 3: REAL-TIME DP (50 episodes)")
print(f"{'─' * 70}")
print(f"  Total updates: {updates_rt}")
print(f"  States ever visited: {[STATE_NAMES[s] for s in sorted(visited_rt)]}")
print(f"  V* = [Cool={V_rt[COOL]:.4f}, Warm={V_rt[WARM]:.4f}, OH={V_rt[OVERHEATED]:.4f}]")
print(f"  Note: Overheated {'was' if OVERHEATED in visited_rt else 'was NOT'} visited "
      f"(terminal state ends episodes)")

# Summary comparison
print(f"\n{'═' * 70}")
print(f"COMPARISON: ALL METHODS REACH SAME V*")
print(f"{'═' * 70}")
print(f"\n  {'Method':<25} {'V(Cool)':<12} {'V(Warm)':<12} {'# Updates':<15}")
print(f"  {'─'*25} {'─'*12} {'─'*12} {'─'*15}")
print(f"  {'Sync Value Iteration':<25} {V_vi[COOL]:<12.4f} {V_vi[WARM]:<12.4f} {(len(vi_h)-1)*3:<15} (sweeps×states)")
print(f"  {'In-Place VI':<25} {V_ip[COOL]:<12.4f} {V_ip[WARM]:<12.4f} {sweeps_ip*3:<15} (sweeps×states)")
print(f"  {'Prioritized Sweeping':<25} {V_ps[COOL]:<12.4f} {V_ps[WARM]:<12.4f} {updates_ps:<15} (individual)")
print(f"  {'Real-Time DP':<25} {V_rt[COOL]:<12.4f} {V_rt[WARM]:<12.4f} {updates_rt:<15} (along trajectories)")
print(f"\n  Winner: Prioritized Sweeping — fewest updates by focusing on high-error states!")

# Part 2: Model-Free Learning (Actual Interaction)

Now we treat the MDP as a **Black Box**. The agent does NOT have access to the transition model `P`. Instead, it must **interact** with the environment by taking actions and observing what happens.

```
┌──────────────────────────────────────────────────────────────────────┐
│  KEY DIFFERENCE: Planning (DP) vs Learning (MC/TD/Q-Learning)        │
│                                                                      │
│  DP (Part 1):                                                        │
│    - Agent READS P[s][a] directly                                    │
│    - Computes expected values by summing over all transitions        │
│    - No randomness — pure computation                                │
│                                                                      │
│  Model-Free (Part 2):                                                │
│    - Agent calls step(s, a) → gets ONE random outcome                │
│    - Must generate many episodes to estimate expected values         │
│    - Learning from SAMPLES of experience, not the full model         │
└──────────────────────────────────────────────────────────────────────┘
```

## 5. Step Function (The 'Actual' World)

The `step()` function simulates the environment. Given a state and action, it **randomly samples** one outcome according to the transition probabilities — just like a real environment would.

```
step(mdp, Cool, Fast) → randomly returns EITHER:
   (Cool, +2.0, False)   with 50% probability
   (Warm, +2.0, False)   with 50% probability
```

The agent sees only the sampled result, NOT the probability distribution behind it.

In [ ]:
def step(mdp, s, a):
    outcomes = mdp.P[s][a]
    idx = np.random.choice(len(outcomes), p=[o[0] for o in outcomes])
    return outcomes[idx][1:] # (ns, r, done)

## 6. Monte Carlo Prediction

**Goal:** Estimate $V_\pi(s)$ by averaging **actual returns** from many complete episodes.

### Algorithm Flow

```
┌──────────────────────────────────────────────────────────────────────┐
│  MONTE CARLO PREDICTION (Every-Visit, constant-α)                    │
│                                                                      │
│  V(s) ← 0 for all s                                                │
│                                                                      │
│  FOR each episode = 1, 2, ..., N:                                   │
│                                                                      │
│    ┌─ PHASE 1: GENERATE EPISODE ──────────────────────────────────┐ │
│    │  Start at s₀ = Cool                                          │ │
│    │  REPEAT:                                                      │ │
│    │    Sample action a ~ π(·|s)    [flip coin: Slow or Fast]     │ │
│    │    Call step(s, a) → (s', r, done)  [environment responds]   │ │
│    │    Record (s, r) in trajectory                                │ │
│    │    s ← s'                                                     │ │
│    │  UNTIL done = True (reached Overheated terminal state)        │ │
│    └──────────────────────────────────────────────────────────────┘ │
│                                                                      │
│    ┌─ PHASE 2: COMPUTE RETURNS (backwards) ───────────────────────┐ │
│    │  G ← 0                                                       │ │
│    │  FOR each (s, r) in trajectory REVERSED:                     │ │
│    │    G ← r + γ · G                                             │ │
│    │    V(s) ← V(s) + α · [G - V(s)]                            │ │
│    └──────────────────────────────────────────────────────────────┘ │
└──────────────────────────────────────────────────────────────────────┘
```

### How Episodes Are Generated (Examples)

With a random policy (50% Slow, 50% Fast), here are some typical episodes:

```
Episode A (Short — lucky):
  Cool ─[Slow, r=+1]─→ Warm ─[Fast, r=-10]─→ Overheated (DONE)
  Trajectory: [(Cool, +1), (Warm, -10)]
  Length: 2 steps

Episode B (Long — stayed cool):
  Cool ─[Slow, r=+1]─→ Cool ─[Fast, r=+2]─→ Warm ─[Slow, r=+1]─→ Cool ─[Fast, r=+2]─→ Warm ─[Fast, r=-10]─→ OH
  Trajectory: [(Cool,+1), (Cool,+2), (Warm,+1), (Cool,+2), (Warm,-10)]
  Length: 5 steps

Episode C (Very short — immediately overheats):
  Cool ─[Fast, r=+2]─→ Warm ─[Fast, r=-10]─→ Overheated (DONE)
  Trajectory: [(Cool, +2), (Warm, -10)]
  Length: 2 steps
```

### Return Computation (Backwards Pass)

For Episode B above, computing returns working backwards (γ = 0.9):

```
Step 5: (Warm, r=-10)  → G = -10 + 0.9×0 = -10.0    → V(Warm) updated
Step 4: (Cool, r=+2)   → G = +2 + 0.9×(-10) = -7.0  → V(Cool) updated
Step 3: (Warm, r=+1)   → G = +1 + 0.9×(-7) = -5.3   → V(Warm) updated
Step 2: (Cool, r=+2)   → G = +2 + 0.9×(-5.3) = -2.77 → V(Cool) updated
Step 1: (Cool, r=+1)   → G = +1 + 0.9×(-2.77) = -1.49 → V(Cool) updated
```

### Why Backwards?

Going backwards lets us compute G incrementally: $G_t = r_{t+1} + \gamma \cdot G_{t+1}$. Starting from the end where $G_T = 0$, each step just needs one multiply-add.

### Update Rule

$$V(s) \leftarrow V(s) + \alpha \left[ G_t - V(s) \right]$$

- $G_t$ is the **actual observed return** from this episode
- $\alpha = 0.01$ is a small learning rate (we're averaging over thousands of episodes)
- Over many episodes, V(s) converges to the **true expected return** under policy π

In [ ]:
def mc_prediction(mdp, policy, episodes=5000, alpha=0.01):
    V = np.zeros(mdp.num_states)
    for i in range(episodes):
        s, traj, G = COOL, [], 0
        while True:
            a = np.random.choice(mdp.num_actions, p=policy[s])
            ns, r, done = step(mdp, s, a); traj.append((s, r))
            if done: break
            s = ns
        for s, r in reversed(traj): G = r + mdp.gamma * G; V[s] += alpha * (G - V[s])
        if (i+1) % 1000 == 0: print(f"Episode {i+1}: V(Cool)={V[COOL]:.2f}")
    return V
V_mc = mc_prediction(mdp, random_policy)

In [ ]:
# DETAILED EXECUTION TRACE: Monte Carlo
# Watch 5 complete episodes being generated, returns computed, and V updated

def mc_prediction_traced(mdp, policy, num_episodes=5, alpha=0.1):
    """MC Prediction with full trace of episode generation and return computation."""
    np.random.seed(42)  # Fixed seed for reproducible demo
    V = np.zeros(mdp.num_states)
    
    print("=" * 70)
    print("MONTE CARLO PREDICTION — DETAILED TRACE")
    print("=" * 70)
    print(f"\nPolicy: equiprobable (π(Slow|s) = π(Fast|s) = 0.5)")
    print(f"α = {alpha}, γ = {mdp.gamma}")
    print(f"Initial V = [Cool=0, Warm=0, OH=0]\n")
    
    for ep in range(num_episodes):
        print(f"{'═' * 70}")
        print(f"EPISODE {ep + 1}")
        print(f"{'═' * 70}")
        
        # Phase 1: Generate episode
        print(f"\n  PHASE 1 — GENERATE EPISODE:")
        s = COOL
        trajectory = []
        step_num = 0
        
        while True:
            a = np.random.choice(mdp.num_actions, p=policy[s])
            ns, r, done = step(mdp, s, a)
            trajectory.append((s, r))
            step_num += 1
            arrow = "→ TERMINAL" if done else f"→ {STATE_NAMES[ns]}"
            print(f"    t={step_num}: {STATE_NAMES[s]} ─[{ACTION_NAMES[a]}, r={r:+.1f}]─ {arrow}")
            if done:
                break
            s = ns
        
        print(f"\n  Trajectory: {[(STATE_NAMES[s], r) for s, r in trajectory]}")
        print(f"  Episode length: {len(trajectory)} steps")
        
        # Phase 2: Compute returns backwards
        print(f"\n  PHASE 2 — COMPUTE RETURNS (backwards) & UPDATE V:")
        G = 0
        print(f"  {'Step':<6} {'(State, r)':<16} {'G = r + γ×G_prev':<28} {'V update':<40} {'V(new)'}")
        print(f"  {'─'*6} {'─'*16} {'─'*28} {'─'*40} {'─'*10}")
        
        for i, (s_t, r_t) in enumerate(reversed(trajectory)):
            G_old = G
            G = r_t + mdp.gamma * G
            v_old = V[s_t]
            V[s_t] += alpha * (G - V[s_t])
            
            step_label = len(trajectory) - i
            g_calc = f"G = {r_t:+.1f} + {mdp.gamma}×{G_old:.2f} = {G:.2f}"
            v_calc = f"V({STATE_NAMES[s_t]}) += {alpha}×({G:.2f} - {v_old:.2f}) = {V[s_t]:.4f}"
            print(f"  {step_label:<6} ({STATE_NAMES[s_t]:<4}, {r_t:+.1f})    {g_calc:<28} {v_calc}")
        
        print(f"\n  After Episode {ep + 1}: V = [Cool={V[COOL]:.4f}, Warm={V[WARM]:.4f}, OH={V[OVERHEATED]:.4f}]")
    
    print(f"\n{'═' * 70}")
    print(f"FINAL MC ESTIMATE (after {num_episodes} episodes):")
    print(f"  V(Cool) = {V[COOL]:.4f}")
    print(f"  V(Warm) = {V[WARM]:.4f}")
    print(f"\nCOMPARISON WITH DP (exact answer from Policy Evaluation):")
    print(f"  V_dp(Cool) = {V_random[COOL]:.4f}")
    print(f"  V_dp(Warm) = {V_random[WARM]:.4f}")
    print(f"\n  (MC needs ~5000 episodes to converge; these 5 are just for illustration)")
    print(f"{'═' * 70}")

mc_prediction_traced(mdp, random_policy, num_episodes=5, alpha=0.1)

## 7. Q-Learning (Off-Policy TD Control)

**Goal:** Learn the optimal Q-values $Q^*(s, a)$ directly — without needing to follow an optimal policy during learning.

### Algorithm Flow

```
┌──────────────────────────────────────────────────────────────────────┐
│  Q-LEARNING                                                          │
│                                                                      │
│  Q(s, a) ← 0 for all s, a                                          │
│                                                                      │
│  FOR each episode = 1, 2, ..., N:                                   │
│    s ← Cool (start state)                                           │
│                                                                      │
│    ┌─ REPEAT (within episode) ────────────────────────────────────┐ │
│    │                                                              │ │
│    │  ┌─ ACTION SELECTION (ε-greedy) ──────────────────────────┐ │ │
│    │  │  With probability ε: pick RANDOM action (explore)      │ │ │
│    │  │  With probability 1-ε: pick argmax_a Q(s,a) (exploit)  │ │ │
│    │  └────────────────────────────────────────────────────────┘ │ │
│    │                         ↓                                    │ │
│    │  Take action a, observe (s', r, done) from environment      │ │
│    │                         ↓                                    │ │
│    │  ┌─ TD UPDATE (the key step) ─────────────────────────────┐ │ │
│    │  │                                                         │ │ │
│    │  │  TD Target = r + γ · max_a' Q(s', a')                  │ │ │
│    │  │              ↑               ↑                          │ │ │
│    │  │          immediate     best possible future             │ │ │
│    │  │          reward        (GREEDY, regardless of           │ │ │
│    │  │                         what action was taken)           │ │ │
│    │  │                                                         │ │ │
│    │  │  TD Error = TD Target - Q(s, a)                         │ │ │
│    │  │                                                         │ │ │
│    │  │  Q(s,a) ← Q(s,a) + α · TD Error                       │ │ │
│    │  └─────────────────────────────────────────────────────────┘ │ │
│    │                         ↓                                    │ │
│    │  s ← s'                                                      │ │
│    │                                                              │ │
│    └─ UNTIL done = True ─────────────────────────────────────────┘ │
└──────────────────────────────────────────────────────────────────────┘
```

### Key Differences from Monte Carlo

| Feature | Monte Carlo | Q-Learning |
|:--------|:-----------|:-----------|
| **When to update** | After ENTIRE episode completes | After EVERY single step |
| **What's used** | Actual return G (full trajectory) | TD target: r + γ·max Q(s',·) |
| **Bootstrapping** | No (uses real outcomes) | Yes (uses estimated future Q) |
| **Bias vs Variance** | Unbiased, high variance | Biased (bootstrapping), low variance |
| **On/Off-Policy** | On-policy (follows π) | Off-policy (learns optimal while exploring) |

### Why "Off-Policy"?

The **behavior** (ε-greedy, explores randomly 10% of time) differs from the **target** (greedy, always picks max Q). Q-learning learns the optimal policy even while acting sub-optimally — this is what makes it powerful.

### Update Rule

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ \underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{TD Target}} - Q(s, a) \right]$$

In [ ]:
def q_learning(mdp, episodes=10000, alpha=0.1, eps=0.1):
    Q = np.zeros((mdp.num_states, mdp.num_actions))
    for i in range(episodes):
        s = COOL
        while True:
            a = np.random.randint(2) if np.random.rand() < eps else np.argmax(Q[s])
            ns, r, done = step(mdp, s, a)
            Q[s, a] += alpha * (r + mdp.gamma * np.max(Q[ns]) * (not done) - Q[s, a])
            if done: break
            s = ns
    return Q

Q_final = q_learning(mdp)
print("\n--- Q-LEARNING FINAL STRATEGY ---")
data_q = [[STATE_NAMES[s], ACTION_NAMES[np.argmax(Q_final[s])] if s != OVERHEATED else "Terminal"] for s in range(mdp.num_states)]
print(pd.DataFrame(data_q, columns=['State', 'Learned Action']).to_string(index=False))

In [ ]:
# DETAILED EXECUTION TRACE: Q-Learning
# Watch the first 3 episodes step-by-step with Q-table updates

def q_learning_traced(mdp, num_episodes=3, alpha=0.1, eps=0.3):
    """Q-Learning with step-by-step trace showing ε-greedy selection and TD updates."""
    np.random.seed(7)  # Fixed seed for reproducible demo
    Q = np.zeros((mdp.num_states, mdp.num_actions))
    
    print("=" * 70)
    print("Q-LEARNING — DETAILED TRACE")
    print("=" * 70)
    print(f"\nα = {alpha}, γ = {mdp.gamma}, ε = {eps}")
    print(f"Initial Q-table: all zeros")
    print(f"\n  Q-Table:")
    print(f"  {'State':<12} {'Q(·,Slow)':<12} {'Q(·,Fast)':<12} {'Best Action'}")
    print(f"  {'─'*12} {'─'*12} {'─'*12} {'─'*12}")
    for s in range(mdp.num_states):
        best = ACTION_NAMES[np.argmax(Q[s])] if Q[s].any() else "tie"
        print(f"  {STATE_NAMES[s]:<12} {Q[s,SLOW]:<12.4f} {Q[s,FAST]:<12.4f} {best}")
    
    for ep in range(num_episodes):
        print(f"\n{'═' * 70}")
        print(f"EPISODE {ep + 1}")
        print(f"{'═' * 70}")
        
        s = COOL
        step_num = 0
        
        while True:
            step_num += 1
            
            # ε-greedy action selection
            rand_val = np.random.rand()
            if rand_val < eps:
                a = np.random.randint(2)
                selection = f"EXPLORE (rand={rand_val:.3f} < ε={eps})"
            else:
                a = np.argmax(Q[s])
                selection = f"EXPLOIT (rand={rand_val:.3f} ≥ ε={eps}, argmax Q)"
            
            # Take action
            ns, r, done = step(mdp, s, a)
            
            # Compute TD update
            td_target = r + mdp.gamma * np.max(Q[ns]) * (not done)
            td_error = td_target - Q[s, a]
            q_old = Q[s, a]
            Q[s, a] += alpha * td_error
            
            print(f"\n  Step {step_num}: State={STATE_NAMES[s]}")
            print(f"    Action selection: {selection} → {ACTION_NAMES[a]}")
            print(f"    Environment: step({STATE_NAMES[s]}, {ACTION_NAMES[a]}) → "
                  f"({STATE_NAMES[ns]}, r={r:+.1f}, done={done})")
            print(f"    TD Target = r + γ·max_a'Q(s',a') = {r} + {mdp.gamma}×{np.max(Q[ns]):.4f}×{int(not done)} = {td_target:.4f}")
            print(f"    TD Error = {td_target:.4f} - Q({STATE_NAMES[s]},{ACTION_NAMES[a]})={q_old:.4f} = {td_error:.4f}")
            print(f"    Update: Q({STATE_NAMES[s]},{ACTION_NAMES[a]}) = {q_old:.4f} + {alpha}×{td_error:.4f} = {Q[s,a]:.4f}")
            
            if done:
                print(f"    → Episode TERMINATED (reached {STATE_NAMES[ns]})")
                break
            s = ns
        
        print(f"\n  Q-Table after Episode {ep + 1}:")
        print(f"  {'State':<12} {'Q(·,Slow)':<12} {'Q(·,Fast)':<12} {'Best Action'}")
        print(f"  {'─'*12} {'─'*12} {'─'*12} {'─'*12}")
        for s in range(mdp.num_states):
            best = ACTION_NAMES[np.argmax(Q[s])] if Q[s].any() else "tie"
            print(f"  {STATE_NAMES[s]:<12} {Q[s,SLOW]:<12.4f} {Q[s,FAST]:<12.4f} {best}")
    
    print(f"\n{'═' * 70}")
    print(f"Q-LEARNING INSIGHTS:")
    print(f"  - Notice updates happen EVERY STEP (not waiting for episode end)")
    print(f"  - TD Target uses max Q(s',·) regardless of action actually taken")
    print(f"  - After 10000 episodes, converged Q-table gives optimal policy:")
    print(f"    Q_final(Cool,Slow)={Q_final[COOL,SLOW]:.2f}, Q_final(Cool,Fast)={Q_final[COOL,FAST]:.2f} → {ACTION_NAMES[np.argmax(Q_final[COOL])]}")
    print(f"    Q_final(Warm,Slow)={Q_final[WARM,SLOW]:.2f}, Q_final(Warm,Fast)={Q_final[WARM,FAST]:.2f} → {ACTION_NAMES[np.argmax(Q_final[WARM])]}")
    print(f"{'═' * 70}")

q_learning_traced(mdp, num_episodes=3, alpha=0.1, eps=0.3)

## 8. Summary: All Methods Compared

All five algorithms converge to the **same optimal answer** for this MDP: **Go Fast when Cool, go Slow when Warm**.

```
┌────────────────────────────────────────────────────────────────────────────────┐
│                        ALGORITHM COMPARISON                                    │
├───────────────────┬──────────────┬────────────────┬───────────────────────────┤
│ Algorithm         │ Needs Model? │ Update Timing  │ What it Learns            │
├───────────────────┼──────────────┼────────────────┼───────────────────────────┤
│ Policy Evaluation │ YES (P)      │ Synchronous    │ V_π (given fixed π)       │
│ Policy Iteration  │ YES (P)      │ Synchronous    │ π* (optimal policy)       │
│ Value Iteration   │ YES (P)      │ Synchronous    │ V* → π* (optimal values)  │
│ Monte Carlo       │ NO           │ End of episode │ V_π (from samples)        │
│ Q-Learning        │ NO           │ Every step     │ Q* → π* (optimal Q)       │
├───────────────────┼──────────────┼────────────────┼───────────────────────────┤
│                   │              │                │                           │
│ DP Methods        │ "Planning"   │ Exact          │ Use full probability      │
│ (top 3)           │              │ (expected      │ distributions             │
│                   │              │  values)       │                           │
│                   │              │                │                           │
│ Model-Free        │ "Learning"   │ Approximate    │ Use sampled               │
│ (bottom 2)        │              │ (sample-based) │ experiences               │
└───────────────────┴──────────────┴────────────────┴───────────────────────────┘
```

### The Fundamental Trade-off

```
                    EXACT (DP)                    APPROXIMATE (Model-Free)
                    ─────────                     ────────────────────────
Pros:  ✓ Converges in few sweeps       ✓ Works without knowing P
       ✓ No variance                    ✓ Scales to huge state spaces
       ✓ Deterministic result           ✓ Can learn from real interaction

Cons:  ✗ Must know full model P         ✗ Needs many episodes (high variance)
       ✗ Sweeps over ALL states         ✗ Results vary between runs
       ✗ Infeasible for large MDPs      ✗ Slower convergence
```

### In This Notebook

All methods discovered the same optimal policy because the Race Car MDP is small enough for DP to solve exactly AND for MC/Q-Learning to converge with enough episodes. In real-world problems (millions of states), only model-free methods are feasible — which is why they dominate modern RL.